# 🔄 REQUEST Interceptor → Cedar Policy Engine 마이그레이션

이번 노트북을 통해서는 `01-pii-gateway-setup.ipynb`에서 배포한 **이중 인터셉터 기반 Gateway**를
**Cedar Policy Engine + RESPONSE 인터셉터** 구조로 마이그레이션 하는 실습을 진행하고자 합니다.

<div style="background:white;padding:15px;border:1px solid #e0e0e0;border-radius:8px;margin:10px 0;">
<h4 style="margin:0 0 10px 0;color:#333;">전체 시스템 아키텍처</h4>
<img src="generated-diagrams/09-new-architecture-overview.png" alt="전체 시스템 아키텍처" style="max-width:100%;height:auto;" />
</div>

### 마이그레이션 개요

| 구분 | Before (01 노트북) | After (이 노트북) |
|------|--------------------|--------------------|
| **접근 제어** | REQUEST Interceptor Lambda | AgentCore Policy Engine |
| **PII 마스킹** | RESPONSE Interceptor Lambda | RESPONSE Interceptor Lambda (동일) |
| **인터셉터 수** | 2개 (REQUEST + RESPONSE) | 1개 (RESPONSE만) |
| **권한 정의** | DynamoDB EmployeePermissions | Cedar Policy |
| **도구 필터링** | Response Interceptor (tools/list) | Policy Engine + Response Interceptor |


### 변경되는 리소스

| 리소스 | 변경 사항 |
|--------|-----------|
| **Gateway** | ♻️ 1차 업데이트 (REQUEST 인터셉터 제거) |
| **Policy Engine** | 🆕 새로 생성 |
| **Cedar 정책 2개** | 🆕 새로 생성 (HR Manager, Employee) |
| **Gateway** | ♻️ 2차 업데이트 (Policy Engine 연결) |
| **Request Interceptor Lambda** | 🗑️ 삭제 (더 이상 불필요) |


### 전체 요청 흐름 변경

```
[Before] Client → JWT Auth → Gateway
                    ├── ① REQUEST Interceptor Lambda (DynamoDB 권한 조회)
                    ├── ② MCP Tool Lambdas
                    └── ③ RESPONSE Interceptor Lambda (PII 마스킹 + tools/list 필터링)

[After]  Client → JWT Auth → Gateway
                    ├── ① AgentCore Policy Engine (client_id 기반 정책 평가)
                    ├── ② MCP Tool Lambdas
                    └── ③ RESPONSE Interceptor Lambda (PII 마스킹 + tools/list 필터링)
```

<div style="background:white;padding:15px;border:1px solid #e0e0e0;border-radius:8px;margin:10px 0;">
<h4 style="margin:0 0 10px 0;color:#333;">마이그레이션 Before/After 아키텍처 비교</h4>
<img src="generated-diagrams/06-migration-before-after-overview.svg" alt="마이그레이션 Before/After 아키텍처 비교" style="max-width:100%;height:auto;" />
</div>
```

---
## 0. 사전 준비: 기존 배포 정보 로드


In [ ]:
import sys
import json
import time
import os
from pathlib import Path
import boto3
from botocore.exceptions import ClientError

NOTEBOOK_DIR = Path(os.path.abspath('')).resolve()
os.chdir(NOTEBOOK_DIR)

project_root = NOTEBOOK_DIR / 'pii-masking-gateway'
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 기존 배포 정보 로드
deployment_file = project_root / 'deployment_info.json'
if not deployment_file.exists():
    raise FileNotFoundError(
        "deployment_info.json이 없습니다. 먼저 01-pii-gateway-setup.ipynb를 실행하세요."
    )

with open(deployment_file, 'r') as f:
    deployment_info = json.load(f)

print("✓ 기존 배포 정보 로드 완료")
print(f"  Deployment ID: {deployment_info['deployment_id']}")
print(f"  Gateway ID: {deployment_info['gateway_id']}")
print(f"  Gateway URL: {deployment_info['gateway_url']}")
print(f"  Region: {deployment_info['region']}")


In [ ]:
# AWS 클라이언트 초기화
REGION = deployment_info['region']
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
lambda_client = boto3.client('lambda', region_name=REGION)
iam_client = boto3.client('iam')
sts_client = boto3.client('sts')

account_id = sts_client.get_caller_identity()['Account']
print(f"✓ AWS Account ID: {account_id}")
print(f"✓ Region: {REGION}")


---
## 1. 현재 Gateway 상태 확인


In [ ]:
gateway_id = deployment_info['gateway_id']
gateway_response = gateway_client.get_gateway(gatewayIdentifier=gateway_id)

print(f"Gateway Name: {gateway_response['name']}")
print(f"Gateway ID: {gateway_id}")
print(f"Gateway Status: {gateway_response['status']}")
print(f"Gateway URL: {gateway_response.get('gatewayUrl', 'N/A')}")
print(f"Gateway ARN: {gateway_response.get('gatewayArn', 'N/A')}")

interceptors = gateway_response.get('interceptorConfigurations', [])
print(f"\nInterceptor Configurations ({len(interceptors)}개):")
for i, ic in enumerate(interceptors):
    points = ic.get('interceptionPoints', [])
    lambda_arn = ic.get('interceptor', {}).get('lambda', {}).get('arn', 'N/A')
    print(f"  [{i+1}] Points: {points}")
    print(f"      Lambda: {lambda_arn}")

policy_config = gateway_response.get('policyEngineConfiguration')
print(f"\nPolicy Engine: {policy_config or '없음 (마이그레이션 대상)'}")

gateway_arn = gateway_response.get('gatewayArn')
gateway_name = gateway_response['name']
gateway_role_arn = gateway_response['roleArn']


---
## 2. Policy Engine 생성


In [ ]:
deployment_id = deployment_info['deployment_id']
policy_engine_name = f"hr_policy_engine_{deployment_id.replace('-', '_')}"

print(f"Creating Policy Engine: {policy_engine_name}")

try:
    pe_response = gateway_client.create_policy_engine(
        name=policy_engine_name,
        description="Policy engine for HR Gateway - migrated from REQUEST interceptor"
    )
    policy_engine_id = pe_response['policyEngineId']
    print(f"✓ Policy Engine created: {policy_engine_id}")
except ClientError as e:
    if e.response['Error']['Code'] == 'ConflictException':
        print(f"  Policy Engine '{policy_engine_name}' already exists, finding it...")
        engines = gateway_client.list_policy_engines(maxResults=100)
        for engine in engines.get('policyEngines', []):
            if engine['name'] == policy_engine_name:
                policy_engine_id = engine['policyEngineId']
                print(f"✓ Found existing Policy Engine: {policy_engine_id}")
                break
    else:
        raise

# ACTIVE 상태까지 대기
print("\n⏳ Policy Engine ACTIVE 대기...")
for attempt in range(30):
    pe_info = gateway_client.get_policy_engine(policyEngineId=policy_engine_id)
    pe_status = pe_info.get('status', 'UNKNOWN')
    policy_engine_arn = pe_info.get('policyEngineArn')
    if pe_status in ('READY', 'ACTIVE', 'Active'):
        print(f"✓ Policy Engine is {pe_status}")
        print(f"  ARN: {policy_engine_arn}")
        break
    print(f"  [{attempt+1}/30] Status: {pe_status}")
    time.sleep(5)
else:
    raise TimeoutError("Policy Engine이 ACTIVE 상태가 되지 않았습니다.")


---
## 3. Gateway에서 REQUEST Interceptor 제거


In [ ]:
print("Gateway에서 REQUEST Interceptor 제거...")
print(f"  Gateway ID: {gateway_id}")

response_lambda_arn = deployment_info['lambda_arn']

update_response = gateway_client.update_gateway(
    gatewayIdentifier=gateway_id,
    name=gateway_name,
    roleArn=gateway_role_arn,
    protocolType='MCP',
    protocolConfiguration={
        'mcp': {'supportedVersions': ['2025-06-18', '2025-03-26'], 'searchType': 'SEMANTIC'}
    },
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration={
        'customJWTAuthorizer': {
            'discoveryUrl': deployment_info['discovery_url'],
            'allowedClients': list(deployment_info['client_ids'].values())
        }
    },
    interceptorConfigurations=[
        {
            'interceptor': {'lambda': {'arn': response_lambda_arn}},
            'interceptionPoints': ['RESPONSE'],
            'inputConfiguration': {'passRequestHeaders': True}
        }
    ],
    exceptionLevel='DEBUG'
)

print("\n⏳ Gateway READY 대기...")
for attempt in range(30):
    gw = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    status = gw.get('status', 'UNKNOWN')
    if status == 'READY':
        ics = gw.get('interceptorConfigurations', [])
        print(f"\n✓ Gateway READY - Interceptors: {len(ics)}개 (RESPONSE만 유지)")
        for ic in ics:
            print(f"    - {ic.get('interceptionPoints')}")
        break
    elif status == 'FAILED':
        reasons = gw.get('statusReasons', [])
        print(f"\n✗ Gateway update FAILED: {reasons}")
        break
    print(f"  [{attempt+1}/30] Status: {status}")
    time.sleep(10)
else:
    print("\n⚠ Timeout waiting for Gateway")


---
## 4. Cedar 정책 생성 (client_id 기반)

### Cedar 정책의 Principal 매칭 방식

AgentCore Gateway는 JWT의 `sub` 클레임을 Cedar `principal`의 `id`로 사용합니다 ([Policy scope - Principal](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy-scope.html#policy-principal), [Schema constraints - Principal Type](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy-schema-constraints.html)).

Cognito `client_credentials` 플로우에서 발급된 access token의 `sub` 클레임은 `client_id` 값과 동일합니다.
따라서 Cedar 정책에서 `client_id`로 직접 principal을 매칭할 수 있습니다.

```
Cognito client_credentials → JWT { sub: "abc123..." } → Cedar principal == AgentCore::OAuthUser::"abc123..."
```

| 정책 | 대상 (client_id) | 허용 도구 |
|------|-------------------|-----------|
| HR Manager Policy | hr-manager client_id | all_employees, employee_search, all_leave_records, employee_leave (4개) |
| Employee Policy | employee client_id | employee_search, employee_leave (2개) |

<div style="background:white;padding:15px;border:1px solid #e0e0e0;border-radius:8px;margin:10px 0;">
<h4 style="margin:0 0 10px 0;color:#333;">Cedar Policy Engine 평가 흐름</h4>
<img src="generated-diagrams/07-cedar-policy-evaluation-flow-overview.svg" alt="Cedar Policy Engine 평가 흐름" style="max-width:100%;height:auto;" />
</div>


In [ ]:
# Action 이름 매핑
tool_names = [
    'all_employees_tool',
    'employee_search_tool',
    'all_leave_records_tool',
    'employee_leave_tool'
]

target_tool_map = {}
for tool_name in tool_names:
    target_name = f"{tool_name.replace('_', '-')}-target"
    action_name = f"{target_name}___{tool_name}"
    target_tool_map[tool_name] = action_name

print("Action Names:")
for tool_name, action_name in target_tool_map.items():
    print(f"  {tool_name} → {action_name}")

# Cognito client_id
hr_client_id = deployment_info['client_ids']['hr-manager']
emp_client_id = deployment_info['client_ids']['employee']
print(f"\nClient IDs:")
print(f"  HR Manager: {hr_client_id}")
print(f"  Employee:   {emp_client_id}")

# HR Manager Cedar 정책
hr_actions = [target_tool_map[t] for t in tool_names]
hr_actions_str = ',\n    '.join(f'AgentCore::Action::"{a}"' for a in hr_actions)

hr_manager_cedar = f"""permit(
    principal == AgentCore::OAuthUser::"{hr_client_id}",
    action in [
    {hr_actions_str}
    ],
    resource == AgentCore::Gateway::"{gateway_arn}"
);"""

print("\n" + "="*60)
print("HR Manager Cedar Policy:")
print("="*60)
print(hr_manager_cedar)

# Employee Cedar 정책
emp_actions = [target_tool_map['employee_search_tool'], target_tool_map['employee_leave_tool']]
emp_actions_str = ',\n    '.join(f'AgentCore::Action::"{a}"' for a in emp_actions)

employee_cedar = f"""permit(
    principal == AgentCore::OAuthUser::"{emp_client_id}",
    action in [
    {emp_actions_str}
    ],
    resource == AgentCore::Gateway::"{gateway_arn}"
);"""

print("\n" + "="*60)
print("Employee Cedar Policy:")
print("="*60)
print(employee_cedar)


In [ ]:
# Policy Engine에 Cedar 정책 등록
policy_suffix = deployment_id.replace('-', '_')

def create_or_update_policy(engine_id, name, cedar_statement, description=""):
    """Cedar 정책 생성 또는 업데이트. 실패 상태 정책은 삭제 후 재생성."""
    policy_def = {'cedar': {'statement': cedar_statement}}

    try:
        response = gateway_client.create_policy(
            policyEngineId=engine_id,
            name=name,
            description=description,
            definition=policy_def
        )
        policy_id = response['policyId']
        print(f"✓ Policy created: {name} (ID: {policy_id})")
        return policy_id
    except ClientError as e:
        if e.response['Error']['Code'] != 'ConflictException':
            raise

    # 이미 존재하는 정책 찾기
    print(f"  Policy '{name}' already exists, checking status...")
    policies = gateway_client.list_policies(
        policyEngineId=engine_id, maxResults=100
    ).get('policies', [])

    for p in policies:
        if p['name'] != name:
            continue
        pid = p['policyId']
        status = p.get('status', '')

        # 실패 상태 → 삭제 후 재생성
        if status in ('CREATE_FAILED', 'UPDATE_FAILED'):
            print(f"  ⚠ Status={status}, 삭제 후 재생성...")
            gateway_client.delete_policy(policyEngineId=engine_id, policyId=pid)
            time.sleep(5)
            response = gateway_client.create_policy(
                policyEngineId=engine_id,
                name=name,
                description=description,
                definition=policy_def
            )
            print(f"✓ Policy recreated: {name} (ID: {response['policyId']})")
            return response['policyId']

        # ACTIVE → 업데이트
        gateway_client.update_policy(
            policyEngineId=engine_id,
            policyId=pid,
            definition=policy_def,
            description=description
        )
        print(f"✓ Policy updated: {name} (ID: {pid})")
        return pid

    raise RuntimeError(f"Policy '{name}' conflict but not found in list")

def wait_for_policy_active(engine_id, policy_id, policy_name, max_attempts=30):
    """정책이 ACTIVE 상태가 될 때까지 대기"""
    for attempt in range(max_attempts):
        try:
            p = gateway_client.get_policy(policyEngineId=engine_id, policyId=policy_id)
            status = p.get('status', 'UNKNOWN')
            if status in ('ACTIVE', 'Active', 'READY'):
                print(f"  ✓ {policy_name} is {status}")
                return True
            if status in ('CREATE_FAILED', 'UPDATE_FAILED'):
                reasons = p.get('statusReasons', [])
                print(f"  ✗ {policy_name} FAILED: {reasons}")
                return False
            print(f"  [{attempt+1}/{max_attempts}] {policy_name}: {status}")
        except Exception as e:
            print(f"  [{attempt+1}/{max_attempts}] {policy_name}: {e}")
        time.sleep(3)
    print(f"  ⚠ Timeout waiting for {policy_name}")
    return False

# HR Manager 정책
hr_policy_name = f"hr_manager_policy_{policy_suffix}"
hr_policy_id = create_or_update_policy(
    policy_engine_id, hr_policy_name, hr_manager_cedar,
    "HR Manager - access to all 4 HR tools (client_id based)"
)

# Employee 정책
emp_policy_name = f"employee_policy_{policy_suffix}"
emp_policy_id = create_or_update_policy(
    policy_engine_id, emp_policy_name, employee_cedar,
    "Employee - limited access to search and leave tools (client_id based)"
)

# 정책 ACTIVE 대기
print("\n⏳ 정책 ACTIVE 대기...")
wait_for_policy_active(policy_engine_id, hr_policy_id, hr_policy_name)
wait_for_policy_active(policy_engine_id, emp_policy_id, emp_policy_name)

# 등록된 정책 확인
all_policies = gateway_client.list_policies(
    policyEngineId=policy_engine_id, maxResults=100
).get('policies', [])
print(f"\n✓ Policy Engine에 등록된 정책: {len(all_policies)}개")
for p in all_policies:
    print(f"  - {p['name']} (Status: {p.get('status', 'N/A')})")


---
## 5. Gateway Role에 Policy Engine 권한 추가

Gateway Execution Role에 Policy Engine 접근 권한을 추가합니다.

| 권한 | 용도 |
|------|------|
| `bedrock-agentcore:GetPolicyEngine` | Policy Engine 설정 조회 |
| `bedrock-agentcore:AuthorizeAction` | Cedar 정책 기반 인가 결정 |
| `bedrock-agentcore:PartiallyAuthorizeActions` | 호출 가능한 도구 목록 조회 |

> 📖 [AWS 문서 - AgentCore Gateway and Policy IAM Permissions](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/policy-permissions.html)


In [ ]:
gateway_role_name = gateway_role_arn.split('/')[-1]
print(f"Gateway Role: {gateway_role_name}")

policy_engine_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "PolicyEngineConfiguration",
            "Effect": "Allow",
            "Action": ["bedrock-agentcore:GetPolicyEngine"],
            "Resource": [f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:policy-engine/*"]
        },
        {
            "Sid": "PolicyEngineAuthorization",
            "Effect": "Allow",
            "Action": [
                "bedrock-agentcore:AuthorizeAction",
                "bedrock-agentcore:PartiallyAuthorizeActions"
            ],
            "Resource": [
                f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:policy-engine/*",
                f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:gateway/*"
            ]
        }
    ]
}

iam_client.put_role_policy(
    RoleName=gateway_role_name,
    PolicyName='PolicyEngineAccess',
    PolicyDocument=json.dumps(policy_engine_policy)
)
print(f"✓ Policy Engine 권한 추가 완료")

print("\n⏳ IAM 정책 전파 대기 (10초)...")
time.sleep(10)
print("✓ 완료")


---
## 6. Gateway에 Policy Engine 연결


In [ ]:
print("Updating Gateway...")
print(f"  Gateway ID: {gateway_id}")
print(f"  변경 사항:")
print(f"    - Policy Engine: 연결 (ENFORCE)")

response_lambda_arn = deployment_info['lambda_arn']

update_response = gateway_client.update_gateway(
    gatewayIdentifier=gateway_id,
    name=gateway_name,
    roleArn=gateway_role_arn,
    protocolType='MCP',
    protocolConfiguration={
        'mcp': {'supportedVersions': ['2025-06-18', '2025-03-26'], 'searchType': 'SEMANTIC'}
    },
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration={
        'customJWTAuthorizer': {
            'discoveryUrl': deployment_info['discovery_url'],
            'allowedClients': list(deployment_info['client_ids'].values())
        }
    },
    interceptorConfigurations=[
        {
            'interceptor': {'lambda': {'arn': response_lambda_arn}},
            'interceptionPoints': ['RESPONSE'],
            'inputConfiguration': {'passRequestHeaders': True}
        }
    ],
    policyEngineConfiguration={
        'arn': policy_engine_arn,
        'mode': 'ENFORCE'
    },
    exceptionLevel='DEBUG'
)

print(f"\n✓ Gateway 업데이트 요청 완료")

print("\n⏳ Gateway READY 대기...")
for attempt in range(30):
    gw = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
    status = gw.get('status', 'UNKNOWN')
    print(f"  [{attempt+1}/30] Status: {status}")
    if status == 'READY':
        print(f"\n✓ Gateway is READY!")
        print(f"  URL: {gw.get('gatewayUrl')}")
        pe_config = gw.get('policyEngineConfiguration')
        if pe_config:
            print(f"  Policy Engine: {pe_config.get('arn')}")
            print(f"  Mode: {pe_config.get('mode')}")
        ics = gw.get('interceptorConfigurations', [])
        print(f"  Interceptors: {len(ics)}개")
        for ic in ics:
            print(f"    - {ic.get('interceptionPoints')}")
        break
    elif status == 'FAILED':
        reasons = gw.get('statusReasons', [])
        print(f"\n✗ Gateway update FAILED: {reasons}")
        break
    time.sleep(10)
else:
    print("\n⚠ Timeout waiting for Gateway")


### Gateway Observability 활성화 (Logs + Traces)


In [ ]:
logs_client = boto3.client('logs', region_name=REGION)

print("Enabling Gateway Observability (Logs + Traces)")

observability_info = {}

# 1. Log Group
log_group_name = f'/aws/vendedlogs/bedrock-agentcore/{gateway_id}'
try:
    logs_client.create_log_group(logGroupName=log_group_name)
    print(f"✓ Log group: {log_group_name}")
except logs_client.exceptions.ResourceAlreadyExistsException:
    print(f"  Log group 이미 존재: {log_group_name}")

log_group_arn = f'arn:aws:logs:{REGION}:{account_id}:log-group:{log_group_name}'

# 2. Delivery Sources
logs_source_resp = None
try:
    logs_source_resp = logs_client.put_delivery_source(
        name=f"{gateway_id}-logs-source", logType='APPLICATION_LOGS', resourceArn=gateway_arn)
    print(f"✓ Logs delivery source")
except Exception as e:
    print(f"⚠ Logs source: {e}")

traces_source_resp = None
try:
    traces_source_resp = logs_client.put_delivery_source(
        name=f"{gateway_id}-traces-source", logType='TRACES', resourceArn=gateway_arn)
    print(f"✓ Traces delivery source")
except Exception as e:
    print(f"⚠ Traces source: {e}")

# 3. Delivery Destinations
logs_dest_resp = None
try:
    logs_dest_resp = logs_client.put_delivery_destination(
        name=f"{gateway_id}-logs-destination", deliveryDestinationType='CWL',
        deliveryDestinationConfiguration={'destinationResourceArn': log_group_arn})
    print(f"✓ Logs delivery destination")
except Exception as e:
    print(f"⚠ Logs destination: {e}")

traces_dest_resp = None
try:
    traces_dest_resp = logs_client.put_delivery_destination(
        name=f"{gateway_id}-traces-destination", deliveryDestinationType='XRAY')
    print(f"✓ Traces delivery destination")
except Exception as e:
    print(f"⚠ Traces destination: {e}")

# 4. Connect
if logs_source_resp and logs_dest_resp:
    try:
        d = logs_client.create_delivery(
            deliverySourceName=logs_source_resp['deliverySource']['name'],
            deliveryDestinationArn=logs_dest_resp['deliveryDestination']['arn'])
        observability_info['logs_delivery_id'] = d['delivery']['id']
        print(f"✓ Logs delivery: {observability_info['logs_delivery_id']}")
    except Exception as e:
        print(f"⚠ Logs delivery: {e}")

if traces_source_resp and traces_dest_resp:
    try:
        d = logs_client.create_delivery(
            deliverySourceName=traces_source_resp['deliverySource']['name'],
            deliveryDestinationArn=traces_dest_resp['deliveryDestination']['arn'])
        observability_info['traces_delivery_id'] = d['delivery']['id']
        print(f"✓ Traces delivery: {observability_info['traces_delivery_id']}")
    except Exception as e:
        print(f"⚠ Traces delivery: {e}")

print(f"\n✓ Observability 활성화 완료")


---
## 7. 마이그레이션 테스트

| # | 역할 | 도구 | 예상 결과 |
|---|------|------|-----------|
| 1 | HR Manager | `tools/list` | 4개 도구 모두 반환 |
| 2 | Employee | `tools/list` | 2개 도구만 반환 |
| 3 | HR Manager | `all_employees_tool` | ✅ 허용 |
| 4 | Employee | `all_employees_tool` | ❌ 거부 |
| 5 | Employee | `employee_search_tool` | ✅ 허용 (PII 마스킹) |


In [ ]:
import requests
import base64

def get_access_token(token_url, client_id, client_secret):
    resp = requests.post(token_url,
        headers={'Content-Type': 'application/x-www-form-urlencoded'},
        data={'grant_type': 'client_credentials', 'client_id': client_id, 'client_secret': client_secret})
    resp.raise_for_status()
    return resp.json()['access_token']

def decode_jwt_payload(token):
    payload = token.split('.')[1]
    padding = 4 - len(payload) % 4
    if padding != 4:
        payload += '=' * padding
    return json.loads(base64.b64decode(payload))

def call_gateway(gateway_url, token, method, params=None):
    resp = requests.post(gateway_url,
        headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {token}', 'Accept': 'application/json'},
        json={'jsonrpc': '2.0', 'id': 1, 'method': method, 'params': params or {}})
    return resp.status_code, resp.json()

def is_denied(body):
    if 'error' in body:
        msg = json.dumps(body['error']).lower()
        return any(kw in msg for kw in ['denied', 'not allowed', 'forbidden', 'unauthorized', 'unknown tool'])
    return False

# 토큰 발급
gateway_url = deployment_info['gateway_url']
token_url = deployment_info['token_url']
hr_client = deployment_info['clients']['hr-manager']
emp_client = deployment_info['clients']['employee']

hr_token = get_access_token(token_url, hr_client['client_id'], hr_client['client_secret'])
emp_token = get_access_token(token_url, emp_client['client_id'], emp_client['client_secret'])
print("✓ HR Manager token obtained")
print("✓ Employee token obtained")

# JWT sub == client_id 확인
hr_claims = decode_jwt_payload(hr_token)
emp_claims = decode_jwt_payload(emp_token)
print(f"\nHR Manager sub: {hr_claims.get('sub')} (match: {'✅' if hr_claims.get('sub') == hr_client['client_id'] else '❌'})")
print(f"Employee sub:   {emp_claims.get('sub')} (match: {'✅' if emp_claims.get('sub') == emp_client['client_id'] else '❌'})")


In [ ]:
# ── 테스트 실행 ──

print("=" * 60)
print("Test 1: HR Manager → tools/list (4개 예상)")
print("=" * 60)
_, body = call_gateway(gateway_url, hr_token, 'tools/list')
tools = body.get('result', {}).get('tools', [])
print(f"  Tools: {len(tools)}")
for t in tools: print(f"    - {t['name']}")
test1_pass = len(tools) == 4

print()
print("=" * 60)
print("Test 2: Employee → tools/list (2개 예상)")
print("=" * 60)
_, body = call_gateway(gateway_url, emp_token, 'tools/list')
tools = body.get('result', {}).get('tools', [])
print(f"  Tools: {len(tools)}")
for t in tools: print(f"    - {t['name']}")
test2_pass = len(tools) == 2

print()
print("=" * 60)
print("Test 3: HR Manager → all_employees_tool (허용 예상)")
print("=" * 60)
_, body = call_gateway(gateway_url, hr_token, 'tools/call', {
    'name': 'all-employees-tool-target___all_employees_tool', 'arguments': {}})
test3_denied = is_denied(body)
if not test3_denied:
    content = body.get('result', {}).get('content', [])
    preview = str(content[0].get('text', ''))[:200] if content else ''
    print(f"  ✅ ALLOWED - {preview}...")
else:
    print(f"  ❌ DENIED (unexpected)")
test3_pass = not test3_denied

print()
print("=" * 60)
print("Test 4: Employee → all_employees_tool (거부 예상)")
print("=" * 60)
_, body = call_gateway(gateway_url, emp_token, 'tools/call', {
    'name': 'all-employees-tool-target___all_employees_tool', 'arguments': {}})
test4_denied = is_denied(body)
print(f"  {'✅ DENIED (expected)' if test4_denied else '❌ ALLOWED (unexpected)'}")
test4_pass = test4_denied

print()
print("=" * 60)
print("Test 5: Employee → employee_search_tool (허용 + PII 마스킹 예상)")
print("=" * 60)
_, body = call_gateway(gateway_url, emp_token, 'tools/call', {
    'name': 'employee-search-tool-target___employee_search_tool', 'arguments': {'employee_id': 'EMP-001'}})
test5_denied = is_denied(body)
if not test5_denied:
    content = body.get('result', {}).get('content', [])
    preview = str(content[0].get('text', ''))[:300] if content else ''
    print(f"  ✅ ALLOWED - {preview}...")
else:
    print(f"  ❌ DENIED (unexpected)")
test5_pass = not test5_denied

# 결과 요약
print()
print("=" * 60)
results = [
    ("HR Manager tools/list (4개)", test1_pass),
    ("Employee tools/list (2개)", test2_pass),
    ("HR Manager all_employees (허용)", test3_pass),
    ("Employee all_employees (거부)", test4_pass),
    ("Employee employee_search (허용)", test5_pass),
]
passed = sum(1 for _, p in results if p)
for name, p in results:
    print(f"  {'✅' if p else '❌'} {name}")
print(f"\n결과: {passed}/{len(results)} passed")
if passed == len(results):
    print("🎉 모든 테스트 통과!")


---
## 8. deployment_info.json 업데이트


In [ ]:
deployment_info['policy_engine_id'] = policy_engine_id
deployment_info['policy_engine_arn'] = policy_engine_arn
deployment_info['policy_engine_name'] = policy_engine_name
deployment_info['migration_completed'] = True
deployment_info['migration_type'] = 'request_interceptor_to_policy_engine'

if 'request_interceptor_name' in deployment_info:
    deployment_info['old_request_interceptor_name'] = deployment_info.pop('request_interceptor_name')
if 'request_interceptor_arn' in deployment_info:
    deployment_info['old_request_interceptor_arn'] = deployment_info.pop('request_interceptor_arn')

with open(deployment_file, 'w') as f:
    json.dump(deployment_info, f, indent=2, ensure_ascii=False)

print("✓ deployment_info.json 업데이트 완료")
print(f"  policy_engine_id: {policy_engine_id}")
print(f"  migration_completed: True")


---
## 9. (선택) 기존 Request Interceptor Lambda 삭제

> ⚠️ 롤백 가능성이 있다면 삭제하지 않아도 됩니다.


In [ ]:
old_request_interceptor = deployment_info.get('old_request_interceptor_name')

if old_request_interceptor:
    print(f"기존 Request Interceptor: {old_request_interceptor}")
    try:
        lambda_client.delete_function(FunctionName=old_request_interceptor)
        print(f"✓ 삭제 완료")
        deployment_info.pop('old_request_interceptor_name', None)
        deployment_info.pop('old_request_interceptor_arn', None)
        with open(deployment_file, 'w') as f:
            json.dump(deployment_info, f, indent=2, ensure_ascii=False)
    except lambda_client.exceptions.ResourceNotFoundException:
        print(f"  이미 삭제됨")
    except Exception as e:
        print(f"  ⚠ 삭제 실패: {e}")
else:
    print("Request Interceptor Lambda 참조 없음")


---
## 10. 마이그레이션 요약

```
[Before - 01 노트북]
┌─────────────────────────────────────────────────┐
│ Gateway                                         │
│  ├── REQUEST Interceptor (Lambda)               │
│  │    └── DynamoDB EmployeePermissions 조회      │
│  ├── Tool Lambdas (4개)                         │
│  └── RESPONSE Interceptor (Lambda)              │
│       └── Bedrock Guardrails PII 마스킹          │
└─────────────────────────────────────────────────┘

[After - 이 노트북]
┌─────────────────────────────────────────────────┐
│ Gateway                                         │
│  ├── Cedar Policy Engine (선언적 정책)           │
│  │    └── client_id 기반 접근 제어               │
│  ├── Tool Lambdas (4개)                         │
│  └── RESPONSE Interceptor (Lambda)              │
│       └── Bedrock Guardrails PII 마스킹          │
└─────────────────────────────────────────────────┘
```


<div style="background:white;padding:15px;border:1px solid #e0e0e0;border-radius:8px;margin:10px 0;">
<h4 style="margin:0 0 10px 0;color:#333;">마이그레이션 완료 후 최종 아키텍처</h4>
<img src="generated-diagrams/08-migrated-gateway-architecture-overview.svg" alt="마이그레이션 완료 후 최종 아키텍처" style="max-width:100%;height:auto;" />
</div>


---
## 📘 가이드: 새로운 사용자 추가

Policy Engine 환경에서 새로운 사용자를 추가하려면 **기존 4단계 + Cedar 정책 업데이트**가 필요합니다.

### Step 1~4: 기존과 동일

01 노트북의 사용자 추가 가이드와 동일하게 수행합니다:
1. EmployeePermissions 테이블에 권한 추가 (Response Interceptor의 tools/list 필터링 + PII 마스킹에 사용)
2. Employees 테이블에 직원 데이터 추가
3. LeaveRecords 테이블에 휴가 데이터 추가
4. 웹 데모 USERS_DB 업데이트

> `pii-masking-gateway/scripts/add_new_user.py` 스크립트로 Step 1~4를 자동 처리할 수 있습니다.

### Step 5: Cedar 정책에 새 사용자 추가 (Policy Engine 환경 전용)

Policy Engine은 Cognito `client_id` 기반으로 접근 제어를 수행합니다.
새 사용자가 기존 Cognito 앱 클라이언트(hr-manager 또는 employee)를 사용한다면 추가 작업이 필요 없습니다.

새로운 역할의 Cognito 앱 클라이언트를 추가하는 경우에만 Cedar 정책을 생성해야 합니다:

```python
# 새 역할의 Cedar 정책 생성 예시
new_cedar = f"""permit(
    principal == AgentCore::OAuthUser::\"{new_client_id}\",
    action in [
        AgentCore::Action::"employee-search-tool-target___employee_search_tool",
        AgentCore::Action::"employee-leave-tool-target___employee_leave_tool"
    ],
    resource == AgentCore::Gateway::\"{gateway_arn}\"
);"""

gateway_client.create_policy(
    policyEngineId=policy_engine_id,
    name='new_role_policy',
    definition={'cedar': {'statement': new_cedar}}
)
```

> **중요**: Cedar 정책은 Cognito `client_id` 단위로 적용됩니다. 같은 앱 클라이언트를 공유하는 사용자들은 동일한 Cedar 정책이 적용되며, 사용자별 세분화는 Response Interceptor의 EmployeePermissions 조회로 처리됩니다.


---
## 📘 가이드: 새로운 MCP Tool 추가

Policy Engine 환경에서 새로운 도구를 추가하려면 **기존 4단계 + Cedar 정책 업데이트**가 필요합니다.

### Step 1~4: 기존과 동일

01 노트북의 도구 추가 가이드와 동일하게 수행합니다:
1. Lambda 함수 코드 작성
2. Lambda 배포
3. Gateway Target 등록
4. EmployeePermissions에 도구 권한 추가 (Response Interceptor의 tools/list 필터링에 사용)

> `pii-masking-gateway/scripts/add_new_tool.py` 스크립트로 Step 1~4를 자동 처리할 수 있습니다.

### Step 5: Cedar 정책에 새 도구 Action 추가 (Policy Engine 환경 전용)

Cedar는 default deny이므로, 새 도구의 Action을 기존 정책에 추가하지 않으면 호출이 거부됩니다.

```python
# 기존 HR Manager 정책에 새 도구 추가 예시
updated_hr_cedar = f"""permit(
    principal == AgentCore::OAuthUser::\"{hr_client_id}\",
    action in [
        AgentCore::Action::"all-employees-tool-target___all_employees_tool",
        AgentCore::Action::"employee-search-tool-target___employee_search_tool",
        AgentCore::Action::"all-leave-records-tool-target___all_leave_records_tool",
        AgentCore::Action::"employee-leave-tool-target___employee_leave_tool",
        AgentCore::Action::"my-new-tool-target___my_new_tool"
    ],
    resource == AgentCore::Gateway::\"{gateway_arn}\"
);"""

gateway_client.update_policy(
    policyEngineId=policy_engine_id,
    policyId=hr_policy_id,
    definition={'cedar': {'statement': updated_hr_cedar}}
)
```

### Action 이름 규칙

Gateway Target으로 등록된 도구의 Action 이름은 `{target_name}___{tool_name}` 형식입니다:

| Target 이름 | Tool 이름 | Cedar Action |
|-------------|-----------|-------------|
| `my-new-tool-target` | `my_new_tool` | `my-new-tool-target___my_new_tool` |

> **중요**: Step 4 (EmployeePermissions)와 Step 5 (Cedar 정책) 모두 수행해야 합니다. Cedar 정책은 Gateway 레벨의 접근 제어를, EmployeePermissions는 Response Interceptor의 tools/list 필터링과 PII 마스킹 역할 판별에 사용됩니다.
